### simular

In [1]:
import sys 
sys.path.append('C:/Users/DATA/Documents/datos/01_script/inicio/funciones')
from funciones import *
from funciones_spark import *
from variables_inicio import *
from utils_sql import *

spark = SparkSession.builder \
    .appName("SparkExample") \
    .master("local[*]") \
    .config('spark.driver.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.2.1.jre11.jar') \
    .config('spark.executor.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.2.1.jre11.jar') \
    .config('spark.executor.memory', '8g') \
    .config('spark.driver.memory', '8g') \
    .getOrCreate()

In [7]:
query = """
    select  NUMERO_DOCUMENTO from DANTALION.dbo.Base_Maestra_ALFIN_BK
    where Fecha_Envio>='2026-08-01'

    """
df_formato=obtener_tabla_sql(spark,query,server_kishin,user_kishin,pwd_kishin,db_kishin)

In [2]:
query = """
    select NUMERO_DOCUMENTO as DNI,* from DANTALION.dbo.Base_Maestra_ALFIN_BK
    where cl_telf1<>0
    and cl_telf1 is not null
    and fecha_envio='2026-08-01'
    and lote<>'inventario'
    """
df_formato=obtener_tabla_sql(spark,query,server_kishin,user_kishin,pwd_kishin,db_kishin)
query = """
    select distinct Dni
    from SAMANTHA..tmp_llamadas_mes a
    where Numero_Campana in('401','403')
    and Fecha_Llamada>='2026-08-01'    
    """
df_quitar=obtener_tabla_sql(spark,query,server_zeus,user_zeus,pwd_zeus,db_zeus)
# df_ref_1=df_ref_1.join(df_formato,['Dni'],'inner')
df_formato=df_formato.join(df_quitar,['Dni'],'leftanti')
# print(df_ref_1.columns)
df_formato.count()


60351

In [3]:
import math

df_formato = df_formato.limit(
    math.ceil(15000)
)

In [9]:
df_formato.select('lote').distinct().show()

+----------+
|      lote|
+----------+
|NO CLIENTE|
|INVENTARIO|
+----------+



In [ ]:
df_formato=df_formato.filter(F.col('lote').isin(''))

In [ ]:
# append_table_SQL(spark,df_correo,f'Base_Maestra_Alfin_bk_Vigente',server_zeus,user_zeus,pwd_zeus,'SAMANTHA')


In [10]:
append_table_SQL(spark,df_formato,f'borrar_prueba',server_kishin,user_kishin,pwd_kishin,'DANTALION')


ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "c:\Users\DATA\AppData\Local\Programs\Python\Python311\Lib\site-packages\py4j\clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\DATA\AppData\Local\Programs\Python\Python311\Lib\socket.py", line 706, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
ConnectionResetError: [WinError 10054] Se ha forzado la interrupción de una conexión existente por el host remoto

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "c:\Users\DATA\AppData\Local\Programs\Python\Python311\Lib\site-packages\py4j\java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\DATA\AppData\Local\Programs\Python\Python311\Lib\site-pa

Error al insertar JDBC: An error occurred while calling o98.jdbc


In [ ]:
query = """
    select NUMERO_DOCUMENTO as DNI,* from DANTALION.dbo.borrar_prueba
    """
df_formato=obtener_tabla_sql(spark,query,server_kishin,user_kishin,pwd_kishin,db_kishin)

In [2]:
df_correo1=cargar_archivo_csv(spark,'aaaaaa_s.csv',';',True)


In [3]:
df_correo1.columns

['_c0', 'dni_cliente', 'celular']

In [4]:

def completar_dni(df):
    return df.withColumn(
        "dni_cliente",
        F.lpad(F.col("dni_cliente").cast("string"), 8, "0")
    )

df_correo1 = completar_dni(df_correo1)

In [28]:
df_correo1=cargar_archivo_csv_ruta(spark,'subir_correo1.csv',';',True,ruta_alfin)
df_correo=cargar_archivo_csv_ruta(spark,'subir_correo.csv',';',True,ruta_alfin)
df_ventas=cargar_archivo_csv_ruta(spark,'TARGET.txt','|',True,ruta_alfin)
df_ventas=df_ventas.filter(F.col('FECHA_DESEMBOLSOS').isin('2026-08-13','2026-08-13'))
df_ventas=df_ventas.select('DNI')
df_correo=df_correo.withColumnRenamed('dni_cliente','DNI')
df_correo1=df_correo1.withColumnRenamed('dni_cliente','DNI')

In [5]:
print(df_formato.columns)

['DNI', 'TIPO_DOI', 'NUMERO_DOCUMENTO', 'NOMBRES', 'APELLIDO_PATERNO', 'APELLIDO_MATERNO', 'SUCURSAL', 'TIENDA', 'DEPARTAMENTO', 'PROVINCIA', 'DISTRITO', 'FEC_NACIMIENTO', 'OFERTA_MAX', 'OFERTA_REEN', 'Tipo_verificacion', 'GRUPO_RIESGO', 'proveedor', 'lote', 'RETIRO', 'Tasa_1', 'Tasa_2', 'Tasa_3', 'Tasa_4', 'Tasa_5', 'Tasa_6', 'Tasa_7', 'segmento', 'Campana', 'PLAZO', 'TEM', 'PROPENSION_IC', 'Desgravamen', 'CUOTA', 'Edad', 'Oferta_12M', 'Tasa_12M', 'Desgravamen_12M', 'CUOTA_12M', 'Oferta_18M', 'Tasa_18M', 'Desgravamen_18M', 'CUOTA_18M', 'Oferta_24M', 'Tasa_24M', 'Desgravamen_24M', 'CUOTA_24M', 'Oferta_36M', 'Tasa_36M', 'Desgravamen_36M', 'CUOTA_36M', 'Validador_Telefono', 'Prioridad', 'Nombre_prioridad', 'Deuda_1', 'Entidad_1', 'Deuda_2', 'Entidad_2', 'Deuda_3', 'Entidad_3', 'sucursal_comercial', 'Agencia_comercial', 'Region_comercial', 'Ubicacion', 'OfertaMaximaSinSeguro', 'color', 'color_final', 'PROPENSION', 'OFERTA_FINAL', 'GARANTIA', 'Oferta_Minima_Paperless', 'RANGO_OFERTA', 'RAN

In [7]:
df_correo.count()

1388

In [29]:

def completar_dni(df):
    return df.withColumn(
        "DNI",
        F.lpad(F.col("DNI").cast("string"), 8, "0")
    )

df_correo = completar_dni(df_correo)
df_ventas = completar_dni(df_ventas)

In [30]:
df_formato = completar_dni(df_formato)
df_inventario = completar_dni(df_inventario)


In [31]:
print(df_ventas.columns)
print(df_correo.columns)
print(df_inventario.columns)

['DNI']
['_c0', 'id', 'canal_campo', 'supervisor', 'ejecutivo_target', 'codigo_ejecutivo_id', 'cdv_alfin_banco', 'DNI', 'nombre_cliente', 'color', 'monto_solicitado', 'celular', 'agencia_atencion', 'fecha_visita', 'hora_visita', 'fecha_envio', 'intentos_realizados', 'estado', 'tipo_carga', 'fecha_registro', 'fecha_dia', 'correo_envio', 'cantidad_repeticiones']
['DNI', 'CANAL', 'TIENDA', 'nombre', 'FECHA', 'celular', 'COLOR_FINAL', 'COD_USER_V3', 'USER_V3', 'PERFIL_RO', 'campaña', 'OFERTA_MAX', 'PLAZO', 'CAPACIDAD_MAX', 'FRESCURA', 'rango_deuda', 'numentidades', 'TOTAL_A_LIQUIDAR', 'TASA_CREDITO_ANTERIOR', 'TASA_1', 'TASA_2', 'TASA_3', 'TASA_4', 'TASA_5', 'TASA_6', 'TASA_7', 'MGNEG', 'MARCA_PD', 'AUTORIZACION_DATOS', 'FLAG_DEUDA_V_OFERTA', 'GRUPO_TASA', 'TIPO_BASE', 'PROPENSION_DISTRIBUCION', 'OFERTA_SS', 'TASA_1_SS', 'TASA_2_SS', 'TASA_3_SS', 'TASA_4_SS', 'TASA_5_SS', 'TASA_6_SS', 'TASA_7_SS', 'ALERTA_MAQUETA', 'FEN', 'PERFIL_ESPECIAL', 'TIPO', 'tipo_archivo']


In [8]:
df_correo.show(2)

+------+--------------------+--------------------+----------------+-------------------+---------------+--------+--------------------+-----------+----------------+---------+----------------+------------+---------------+-------------------+-------------------+-------+----------+-------------------+----------+------------+
|    id|         canal_campo|          supervisor|ejecutivo_target|codigo_ejecutivo_id|cdv_alfin_banco|     DNI|      nombre_cliente|      color|monto_solicitado|  celular|agencia_atencion|fecha_visita|    hora_visita|        fecha_envio|intentos_realizados| estado|tipo_carga|     fecha_registro| fecha_dia|correo_envio|
+------+--------------------+--------------------+----------------+-------------------+---------------+--------+--------------------+-----------+----------------+---------+----------------+------------+---------------+-------------------+-------------------+-------+----------+-------------------+----------+------------+
|221918|CALL CENTER / TAR...|CARLO

In [32]:
# df_correo2=df_correo.filter(F.col('fecha_dia').isin('2026-08-05','2026-08-11'))

# df_correo2=df_correo2.filter((F.col('fecha_dia').isin('2026-08-04','2026-08-04'))
#                                 )
# df_correo2=df_correo2.filter((F.col('fecha_dia').isin('2026-08-05','2026-08-11'))
                                # )
df_correo2=df_correo.dropDuplicates(['DNI'])
df_correo2.count()

1388

In [21]:
df_ventas.show()

+--------+
|     DNI|
+--------+
|02604479|
+--------+



In [19]:
df_correo2.join(df_ventas,['DNI'],'inner').count()

0

In [11]:
df_correo2.count()

2598

In [41]:
df_inventario.count()


45713

In [42]:
import math

df_inventario = df_inventario.limit(
    math.ceil(1360 * 2.1)
)
df_inventario.count()

2856

In [11]:
print(df_correo1.columns)
print(df_inventario.columns)
print(df_formato.columns)


['id', 'canal_campo', 'supervisor', 'ejecutivo_target', 'codigo_ejecutivo_id', 'cdv_alfin_banco', 'DNI', 'nombre_cliente', 'color', 'monto_solicitado', 'celular', 'agencia_atencion', 'fecha_visita', 'hora_visita', 'fecha_envio', 'intentos_realizados', 'estado', 'tipo_carga', 'fecha_registro', 'fecha_dia', 'correo_envio']
['DNI', 'CANAL', 'TIENDA', 'nombre', 'FECHA', 'celular', 'COLOR_FINAL', 'COD_USER_V3', 'USER_V3', 'PERFIL_RO', 'campaña', 'OFERTA_MAX', 'PLAZO', 'CAPACIDAD_MAX', 'FRESCURA', 'rango_deuda', 'numentidades', 'TOTAL_A_LIQUIDAR', 'TASA_CREDITO_ANTERIOR', 'TASA_1', 'TASA_2', 'TASA_3', 'TASA_4', 'TASA_5', 'TASA_6', 'TASA_7', 'MGNEG', 'MARCA_PD', 'AUTORIZACION_DATOS', 'FLAG_DEUDA_V_OFERTA', 'GRUPO_TASA', 'TIPO_BASE', 'PROPENSION_DISTRIBUCION', 'OFERTA_SS', 'TASA_1_SS', 'TASA_2_SS', 'TASA_3_SS', 'TASA_4_SS', 'TASA_5_SS', 'TASA_6_SS', 'TASA_7_SS', 'ALERTA_MAQUETA', 'FEN', 'PERFIL_ESPECIAL', 'TIPO', 'tipo_archivo']
['DNI', 'TIPO_DOI', 'NUMERO_DOCUMENTO', 'NOMBRES', 'APELLIDO_PATE

In [ ]:
['DNI', 'TIPO_DOI', 'NUMERO_DOCUMENTO', 'NOMBRES', 'APELLIDO_PATERNO', 'APELLIDO_MATERNO', 'SUCURSAL', 'TIENDA', 'DEPARTAMENTO', 'PROVINCIA', 'DISTRITO', 'FEC_NACIMIENTO', 'OFERTA_MAX', 'OFERTA_REEN', 'Tipo_verificacion', 'GRUPO_RIESGO', 'proveedor', 'lote', 'RETIRO', 'Tasa_1', 'Tasa_2', 'Tasa_3', 'Tasa_4', 'Tasa_5', 'Tasa_6', 'Tasa_7', 'segmento', 'Campana', 'PLAZO', 'TEM', 'PROPENSION_IC', 'Desgravamen', 'CUOTA', 'Edad', 'Oferta_12M', 'Tasa_12M', 'Desgravamen_12M', 'CUOTA_12M', 'Oferta_18M', 'Tasa_18M', 'Desgravamen_18M', 'CUOTA_18M', 'Oferta_24M', 'Tasa_24M', 'Desgravamen_24M', 'CUOTA_24M', 'Oferta_36M', 'Tasa_36M', 'Desgravamen_36M', 'CUOTA_36M', 'Validador_Telefono', 'Prioridad', 'Nombre_prioridad', 'Deuda_1', 'Entidad_1', 'Deuda_2', 'Entidad_2', 'Deuda_3', 'Entidad_3', 'sucursal_comercial', 'Agencia_comercial', 'Region_comercial', 'Ubicacion', 'OfertaMaximaSinSeguro', 'color', 'color_final', 'PROPENSION', 'OFERTA_FINAL', 'GARANTIA', 'Oferta_Minima_Paperless', 'RANGO_OFERTA', 'RANGO_SUELDO', 'CAPACIDAD_MAX', 'PEER', 'PROP_COMER', 'TIPO_GEST', 'CLIENTE_NUEVO', 'GRUPO_TASA', 'NUEVOS_3M', 'NUEVOS_6M', 'NUEVOS_9M', 'NUEVOS_12M', 'NUEVOS_4M', 'GRUPO_MONTO', 'TASA_VS_MONTO', 'USUARIO', 'incremento_monto_riesgos', 'FLG_DEUDA_PLUS', 'tipo_cliente_riegos', 'USER_V3', 'LEAD_CALIDAD', 'SEGMENTO_USER', 'RANGO_EDAD', 'RANGO_OFERTA2', 'PERIODO', 'RETIRO_GEST', 'MEJOR_TIPIFICACION', 'STATUS', 'FECHA_SOL', 'BASE', 'RESULTADO', 'NUM_ENRIQUECIDO', 'TIPO_CONTACTO', 'Q_VENTAS', 'LOCALIDAD', 'DESEMBOLSADO', 'MONTO_DESEMBOLSADO', 'SBI', 'CRUCE', 'PREST_PREVIO', 'ID_CLIENTE', 'RANGO_EDAD2', 'Fecha_Envio', 'TIPO_BD', 'COD_BD', 'NOMB_BD', 'MES_GESTION', 'TIPO_CLIENTE', 'GRUPO_TASA_REENGANCHE', 'SALDO_DIFERENCIAL_REENG', 'FLAG_REENG', 'RETIRO_DESEMBOLSO', 'FRESCURA', 'flag_deuda_v_oferta', 'MGNEG', 'PERFIL_RO', 'TIPO_BASE', 'cl_telf1', 'cl_telf2', 'cl_telf3', 'cl_telf4', 'cl_telf5', 'cl_telf6', 'cl_telf7', 'cl_telf8', 'cl_telf9', 'cl_telf10', 'cl_movil', 'cl_celular', 'cl_telefono', 'cl_turno', 'cl_gestor', 'cl_asesor', 'cl_accion', 'cl_gestion', 'SERVICIO', 'cl_fecha_gestion', 'cl_hora_gestion', 'cl_hits', 'cl_fecha_llamar', 'cl_prioridad', 'cl_orden', 'cl_predictivo', 'cl_tiempo', 'cl_base', 'cl_mes', 'cl_carga', 'id_carga', 'cl_area', 'fecha_alimentacion', 'cl_base_ant', 'cl_accion_ant', 'cl_fecha_ant', 'campania', 'PROMOCION', 'PROMOCION2', 'nombre_base', 'NumEntidades', 'p_banco', 'PERFIL_GLOBAL', 'FLG_AAHH', 'SCORE_TELEFONO', 'PILOTO_PLAZAS', 'INTENSIDAD_MAX', 'marca1', 'marca2', 'marca3', 'AÑO_DURACION_BASE', 'MES_DURACION_BASE', 'FLAT2', 'REP1', 'REP2', 'PILOTO_RETENCION', 'CAMP_BONO', 'ACCION']


In [4]:
df_formato=df_formato.select( 'DNI',  'cl_telf1')
df_formato=df_formato.withColumnRenamed('cl_telf1','celular')

In [34]:
df_correo2=df_correo2.select( 'DNI',  'celular')
df_inventario=df_inventario.select( 'DNI',  'celular')
df_correo2=df_correo2.withColumn('correo_formulario',F.lit(1))
df_inventario=df_inventario.withColumn('correo_formulario',F.lit(2))

In [36]:
df_formato=df_formato.withColumn('correo_formulario',F.lit('3'))

In [43]:
df_dia=df_correo2.unionByName(df_inventario).unionByName(df_formato)


In [38]:
print(df_correo2.columns)
print(df_inventario.columns)
print(df_formato.columns)

['DNI', 'celular', 'correo_formulario']
['DNI', 'celular', 'correo_formulario']
['DNI', 'celular', 'correo_formulario']


In [44]:
df_dia.count()

10403

In [45]:
df_dia=df_dia.dropDuplicates(['DNI'])
# df_dia.count()
# 

In [30]:
df_7500 = df_formato.filter(
    # (F.col('propension_ic').isin('1','2','3'))&
    # (F.col('proveedor').isin(['INVENTARIO TARGET 2', 'CET TARGET 2', 'CET TARGET 0', 'INVENTARIO TARGET 1', 'INVENTARIO TARGET 0', ]))&
    # (F.col('user_v3').isin(user_v3))&

    # (
    (F.col('propension_ic').isin('1','2','3'))&
    (F.col('frescura').isin('0','4'))&
    (F.col('OFERTA_MAX').between(2500,10000))&
    # (F.col('USER_V3').isin('14. Otros Bancarizados','6. MES B','1. sunedu & sunarp A','2. sunedu & sunarp B','3. MES + PLD Peers '))&
    (F.col('retiro')=='ACTIVO')
)
df_7500.count()

11074

In [5]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# df_7500 = df_formato.limit(8560)

# window = Window.orderBy(F.monotonically_increasing_id())

# df_7500 = (
#     df_7500
#     .withColumn(
#         "rn",
#         F.row_number().over(window)
#     )
# )

# # =====================================================
# # 3. ASIGNAR LA FECHA
# # =====================================================
# df_7500 = (
#     df_7500
#     .withColumn(
#         "fecha_envio",
#         F.when(
#             F.col("rn") <= 3750,
#             F.lit("2026-08-01")
#         ).otherwise(
#             F.lit("2026-08-02")
#         )
#     )
#     .drop("rn")
# )
df_7500=df_formato.withColumn('fecha_envio',F.lit('2026-08-18'))

In [ ]:
df_final = df_7500.withColumn(
    "ADDRESS1",
    F.concat_ws(
        " ",
        F.col("NOMBRES"),
        F.col("APELLIDO_PATERNO"),
        F.col("APELLIDO_MATERNO")
    )
)

In [ ]:
w = Window.partitionBy('lote','PROPENSION_IC','FRESCURA','USER_V3').orderBy(F.rand())

df_split = df_final.withColumn("por_dia", F.ntile(1).over(w))

In [6]:
# df_split=df_final.withColumn('fecha_llamada',F.lit('2026-07-29'))
df_split=df_7500.withColumn('Numero_Campana',F.lit('401'))
df_split=df_split.withColumn('list_description',F.lit('provicional'))
df_split=df_split.withColumn('list_name',F.lit('provicional'))
df_split=df_split.withColumn('Nombre_Campana',F.lit('BOT_ALFIN'))
# df_split=df_split.withColumn('Nombre_Campana',F.lit('2026-07 BCO ALFIN'))


In [7]:
df_split=df_split.withColumnRenamed('fecha_envio','fecha_llamada')


In [8]:
df_split=df_split.withColumn('TIPO_LOTE',F.lit('BASE BOT'))


In [9]:
df_split=df_split.withColumn('PROPENSION_IC',F.lit('1'))
df_split=df_split.withColumn('FRESCURA',F.lit('0'))

In [10]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


# ============================================================
# 0. CONFIGURACIÓN GENERAL
# ============================================================

FECHA_COL = "Fecha_Llamada"

# Horario permitido: 09:00 a 17:50
SEGUNDO_INICIO = (9 * 3600) + (0 * 60)
SEGUNDO_FIN = (17 * 3600) + (50 * 60)

# Bot: máximo 15 llamadas simultáneas
MAX_LLAMADAS_BOT = 15

# Separación entre llamadas
GAP_MINIMO = 5
GAP_MAXIMO = 20


# ============================================================
# 1. USUARIOS
# ============================================================

usuarios_regular = [
    "PEC200",
    "PEC188",
    "PEC136",
    "PEC139",
    "PEC184",
    "VDAD"
]

usuarios_bot = [
    "49587612",
    "VDAD"
]

regular_sin_vdad = [
    usuario
    for usuario in usuarios_regular
    if usuario != "VDAD"
]

bot_sin_vdad = [
    usuario
    for usuario in usuarios_bot
    if usuario != "VDAD"
]



# ============================================================
# 2. LIMPIAR TIPOS DE DATOS
# ============================================================

df_split_1 = (
    df_split
    .withColumn(
        FECHA_COL,
        F.to_date(F.col(FECHA_COL))
    )
    .withColumn(
        "PROPENSION_IC",
        F.col("PROPENSION_IC").cast("double")
    )
    .withColumn(
        "FRESCURA",
        F.col("FRESCURA").cast("double")
    )
)



In [11]:


# ============================================================
# 2. LIMPIAR TIPOS DE DATOS
# ============================================================

df_split_1 = (
    df_split_1
    .withColumn(
        FECHA_COL,
        F.to_date(F.col(FECHA_COL))
    )
    .withColumn(
        "PROPENSION_IC",
        F.col("PROPENSION_IC").cast("double")
    )
    .withColumn(
        "FRESCURA",
        F.col("FRESCURA").cast("double")
    )
)


# ============================================================
# 3. PROBABILIDAD DE ASIGNACIÓN A VDAD
# ============================================================

df_split_1 = df_split_1.withColumn(
    "_prob_vdad_base",
    F.when(
        F.col("TIPO_LOTE") == "BASE REGULAR",
        F.lit(0.43)
    )
    .when(
        F.col("TIPO_LOTE") == "BASE BOT",
        F.lit(0.90)
    )
    .otherwise(F.lit(0.43))
)

df_split_1 = df_split_1.withColumn(
    "_score_vdad",
    (
        (
            F.coalesce(F.col("PROPENSION_IC"), F.lit(1.0))
            - F.lit(1.0)
        ) / F.lit(5.0)
        +
        (
            F.coalesce(F.col("FRESCURA"), F.lit(0.0))
            / F.lit(5.0)
        )
    ) / F.lit(2.0)
)

df_split_1 = df_split_1.withColumn(
    "_prob_vdad_final",
    F.least(
        F.col("_prob_vdad_base")
        + F.col("_score_vdad") * F.lit(0.15),
        F.lit(0.99)
    )
)

df_split_1 = df_split_1.withColumn(
    "_rnd_vdad",
    F.rand()
)


In [12]:

# ============================================================
# 3. PROBABILIDAD DE ASIGNACIÓN A VDAD
# ============================================================

df_split_1 = df_split_1.withColumn(
    "_prob_vdad_base",
    F.when(
        F.col("TIPO_LOTE") == "BASE REGULAR",
        F.lit(0.43)
    )
    .when(
        F.col("TIPO_LOTE") == "BASE BOT",
        F.lit(0.92)
    )
    .otherwise(F.lit(0.43))
)

df_split_1 = df_split_1.withColumn(
    "_score_vdad",
    (
        (
            F.coalesce(F.col("PROPENSION_IC"), F.lit(1.0))
            - F.lit(1.0)
        ) / F.lit(5.0)
        +
        (
            F.coalesce(F.col("FRESCURA"), F.lit(0.0))
            / F.lit(5.0)
        )
    ) / F.lit(2.0)
)

df_split_1 = df_split_1.withColumn(
    "_prob_vdad_final",
    F.least(
        F.col("_prob_vdad_base")
        + F.col("_score_vdad") * F.lit(0.15),
        F.lit(0.99)
    )
)

df_split_1 = df_split_1.withColumn(
    "_rnd_vdad",
    F.rand()
)


# ============================================================
# 4. ASIGNAR DNI DEL EJECUTIVO
# ============================================================

df_split_1 = df_split_1.withColumn(
    "DNI_Ejecutivo",
    F.when(
        F.col("_rnd_vdad") <= F.col("_prob_vdad_final"),
        F.lit("VDAD")
    ).otherwise(F.lit(None).cast("string"))
)

array_regular = F.array(
    *[F.lit(usuario) for usuario in regular_sin_vdad]
)

array_bot = F.array(
    *[F.lit(usuario) for usuario in bot_sin_vdad]
)

df_split_1 = df_split_1.withColumn(
    "DNI_Ejecutivo",

    F.when(
        F.col("DNI_Ejecutivo").isNull()
        & (F.col("TIPO_LOTE") == "BASE REGULAR"),

        array_regular[
            F.floor(
                F.rand() * F.lit(len(regular_sin_vdad))
            ).cast("int")
        ]
    )

    .when(
        F.col("DNI_Ejecutivo").isNull()
        & (F.col("TIPO_LOTE") == "BASE BOT"),

        array_bot[
            F.floor(
                F.rand() * F.lit(len(bot_sin_vdad))
            ).cast("int")
        ]
    )

    .otherwise(F.col("DNI_Ejecutivo"))
)


# ============================================================
# 5. ASIGNAR NOMBRE DEL EJECUTIVO
# ============================================================

df_split_1 = df_split_1.withColumn(
    "Ejecutivo",

    F.when(
        (F.col("TIPO_LOTE") == "BASE BOT")
        & (F.col("DNI_Ejecutivo") == "VDAD"),

        F.lit("Outbound Auto Dial")
    )

    .when(
        (F.col("TIPO_LOTE") == "BASE BOT")
        & (F.col("DNI_Ejecutivo") != "VDAD"),

        F.lit("Bot Bco Alfin")
    )

    .when(
        (F.col("TIPO_LOTE") == "BASE REGULAR")
        & F.col("DNI_Ejecutivo").isin(regular_sin_vdad),

        F.lit("Agente BCO Alfin")
    )

    .when(
        (F.col("TIPO_LOTE") == "BASE REGULAR")
        & (F.col("DNI_Ejecutivo") == "VDAD"),

        F.lit("Outbound Auto Dial")
    )

    .otherwise(F.lit(None).cast("string"))
)



In [13]:

# ============================================================
# 6. CÓDIGOS DE PALETA
# ============================================================

codigos_vdad = [
    "PDROP",
    "NA",
    "AB"
]

codigos_regular = [
    "zzz5",
    "zzz6",
    "zzz9",
    "zzz13",
    "zzz10",
    "zzz11",
    "zzz56",
    "zzz23",
    "zzz24",
    "zzz26"
]

codigos_bot = [
    "zzz5",
    "zzz10",
    "zzz19",
    "zzz22",
    "zzz23",
    "zzz15",
    "zzz24",
    "zzz26"
]

array_codigos_vdad = F.array(
    *[F.lit(codigo) for codigo in codigos_vdad]
)

array_codigos_regular = F.array(
    *[F.lit(codigo) for codigo in codigos_regular]
)

array_codigos_bot = F.array(
    *[F.lit(codigo) for codigo in codigos_bot]
)


# ============================================================
# 7. ASIGNAR CÓDIGO DE PALETA
# ============================================================

df_split_1 = df_split_1.withColumn(
    "Codigo_Paleta",

    F.when(
        F.col("DNI_Ejecutivo") == "VDAD",

        array_codigos_vdad[
            F.floor(
                F.rand() * F.lit(len(codigos_vdad))
            ).cast("int")
        ]
    )

    .when(
        (F.col("TIPO_LOTE") == "BASE REGULAR")
        & (F.col("DNI_Ejecutivo") != "VDAD"),

        array_codigos_regular[
            F.floor(
                F.rand() * F.lit(len(codigos_regular))
            ).cast("int")
        ]
    )

    .when(
        (F.col("TIPO_LOTE") == "BASE BOT")
        & (F.col("DNI_Ejecutivo") != "VDAD"),

        array_codigos_bot[
            F.floor(
                F.rand() * F.lit(len(codigos_bot))
            ).cast("int")
        ]
    )

    .otherwise(F.lit(None).cast("string"))
)


# ============================================================
# 8. GENERAR DURACIÓN DE LLAMADA
# ============================================================

df_split_1 = df_split_1.withColumn(
    "segundos",

    # Llamadas sin duración
    F.when(
        F.col("Codigo_Paleta").isin(
            "PDROP",
            "AB",
            "NA",
            "zzz27",
            "zzz26",
            "zzz25",
            "zzz24",
            "zzz23"
        ),
        F.lit(0)
    )

    # Llamada larga: 100 a 150 segundos
    .when(
        F.col("Codigo_Paleta") == "zzz1",
        F.floor(
            F.rand() * F.lit(51) + F.lit(100)
        ).cast("int")
    )

    # Llamada regular: 50 a 100 segundos
    .when(
        F.col("Codigo_Paleta").isin(
            "zzz5",
            "zzz6",
            "zzz9",
            "zzz13",
            "zzz10",
            "zzz11",
            "zzz56"
        ),
        F.floor(
            F.rand() * F.lit(51) + F.lit(50)
        ).cast("int")
    )

    # Llamada BOT: 30 a 80 segundos
    .when(
        F.col("Codigo_Paleta").isin(
            "zzz19",
            "zzz15",
            "zzz22"
        ),
        F.floor(
            F.rand() * F.lit(51) + F.lit(30)
        ).cast("int")
    )

    .otherwise(F.lit(0))
)

df_split_1 = df_split_1.withColumn(
    "_duracion",
    F.coalesce(
        F.col("segundos").cast("int"),
        F.lit(0)
    )
)


# ============================================================
# 9. GENERAR IDENTIFICADOR TEMPORAL
# ============================================================

df_split_1 = df_split_1.withColumn(
    "_id_temporal",
    F.monotonically_increasing_id()
)


# ============================================================
# 10. HORARIO PARA VDAD
# ============================================================
# VDAD recibe una hora aleatoria entre 09:00 y 17:50.
# Se descuenta la duración para que no termine fuera del horario.

df_vdad = (
    df_split_1
    .filter(
        F.col("DNI_Ejecutivo") == "VDAD"
    )
    .withColumn(
        "_ultimo_inicio",
        F.greatest(
            F.lit(SEGUNDO_INICIO),
            F.lit(SEGUNDO_FIN) - F.col("_duracion")
        )
    )
    .withColumn(
        "_segundo_inicio",
        F.floor(
            F.rand()
            * (
                F.col("_ultimo_inicio")
                - F.lit(SEGUNDO_INICIO)
                + F.lit(1)
            )
            + F.lit(SEGUNDO_INICIO)
        ).cast("long")
    )
)


# ============================================================
# 11. HORARIO PARA BASE REGULAR
# ============================================================
# Las llamadas de cada agente se ordenan de manera consecutiva.
# No pueden cruzarse entre sí.

df_regular = df_split_1.filter(
    (F.col("TIPO_LOTE") == "BASE REGULAR")
    & (F.col("DNI_Ejecutivo") != "VDAD")
)

ventana_regular = (
    Window
    .partitionBy(
        FECHA_COL,
        "DNI_Ejecutivo"
    )
    .orderBy(
        F.rand()
    )
)

df_regular = df_regular.withColumn(
    "_orden",
    F.row_number().over(ventana_regular)
)

df_regular = df_regular.withColumn(
    "_gap",
    F.floor(
        F.rand() * F.lit(GAP_MAXIMO - GAP_MINIMO + 1)
        + F.lit(GAP_MINIMO)
    ).cast("int")
)

ventana_acumulada_regular = (
    Window
    .partitionBy(
        FECHA_COL,
        "DNI_Ejecutivo"
    )
    .orderBy(
        "_orden"
    )
    .rowsBetween(
        Window.unboundedPreceding,
        -1
    )
)

df_regular = df_regular.withColumn(
    "_segundo_inicio",

    F.lit(SEGUNDO_INICIO)
    +
    F.coalesce(
        F.sum(
            F.col("_duracion") + F.col("_gap")
        ).over(ventana_acumulada_regular),

        F.lit(0)
    )
)


# ============================================================
# 12. HORARIO PARA BASE BOT
# ============================================================
# Cada grupo de 15 llamadas comparte la misma hora de inicio.

df_bot = df_split_1.filter(
    (F.col("TIPO_LOTE") == "BASE BOT")
    & (F.col("DNI_Ejecutivo") != "VDAD")
)

ventana_bot = (
    Window
    .partitionBy(
        FECHA_COL,
        "DNI_Ejecutivo"
    )
    .orderBy(
        F.rand()
    )
)

df_bot = df_bot.withColumn(
    "_orden",
    F.row_number().over(ventana_bot)
)

df_bot = df_bot.withColumn(
    "_grupo_bot",
    F.floor(
        (F.col("_orden") - F.lit(1))
        / F.lit(MAX_LLAMADAS_BOT)
    ).cast("int")
)

# Duración máxima de cada grupo de 15 llamadas
df_bot_grupos = (
    df_bot
    .groupBy(
        FECHA_COL,
        "DNI_Ejecutivo",
        "_grupo_bot"
    )
    .agg(
        F.max("_duracion").alias("_duracion_grupo")
    )
    .withColumn(
        "_gap_grupo",
        F.floor(
            F.rand() * F.lit(GAP_MAXIMO - GAP_MINIMO + 1)
            + F.lit(GAP_MINIMO)
        ).cast("int")
    )
)

ventana_acumulada_bot = (
    Window
    .partitionBy(
        FECHA_COL,
        "DNI_Ejecutivo"
    )
    .orderBy(
        "_grupo_bot"
    )
    .rowsBetween(
        Window.unboundedPreceding,
        -1
    )
)

df_bot_grupos = df_bot_grupos.withColumn(
    "_segundo_inicio",

    F.lit(SEGUNDO_INICIO)
    +
    F.coalesce(
        F.sum(
            F.col("_duracion_grupo")
            + F.col("_gap_grupo")
        ).over(ventana_acumulada_bot),

        F.lit(0)
    )
)

df_bot = df_bot.join(
    df_bot_grupos.select(
        FECHA_COL,
        "DNI_Ejecutivo",
        "_grupo_bot",
        "_segundo_inicio"
    ),
    on=[
        FECHA_COL,
        "DNI_Ejecutivo",
        "_grupo_bot"
    ],
    how="left"
)


# ============================================================
# 13. UNIR REGULAR, BOT Y VDAD
# ============================================================

df_split_1 = (
    df_regular
    .unionByName(
        df_bot,
        allowMissingColumns=True
    )
    .unionByName(
        df_vdad,
        allowMissingColumns=True
    )
)


# ============================================================
# 14. VALIDAR HORARIO
# ============================================================

df_split_1 = df_split_1.withColumn(
    "_segundo_fin",
    F.col("_segundo_inicio")
    + F.col("_duracion")
)

df_split_1 = df_split_1.withColumn(
    "Dentro_Rango_Horario",

    F.when(
        (F.col("_segundo_inicio") >= F.lit(SEGUNDO_INICIO))
        &
        (F.col("_segundo_fin") <= F.lit(SEGUNDO_FIN)),

        F.lit("SI")
    )

    .otherwise(F.lit("NO"))
)


# ============================================================
# 15. GUARDAR REGISTROS FUERA DEL HORARIO
# ============================================================
# Esto permite revisar qué llamadas no alcanzaron a entrar.

df_fuera_horario = df_split_1.filter(
    F.col("Dentro_Rango_Horario") == "NO"
)


# ============================================================
# 16. CONSERVAR SOLO LLAMADAS DENTRO DEL HORARIO
# ============================================================

df_split_1 = df_split_1.filter(
    F.col("Dentro_Rango_Horario") == "SI"
)


# ============================================================
# 17. CREAR TIMESTAMP BASE DE LA FECHA
# ============================================================

df_split_1 = df_split_1.withColumn(
    "_fecha_base_timestamp",
    F.to_timestamp(
        F.date_format(
            F.col(FECHA_COL),
            "yyyy-MM-dd"
        ),
        "yyyy-MM-dd"
    )
)


# ============================================================
# 18. CREAR FECHA Y HORA DE LLAMADA
# ============================================================
# Se convierte la fecha base a segundos Unix y luego se suman
# los segundos transcurridos desde medianoche.

df_split_1 = df_split_1.withColumn(
    "Fecha_Hora_Llamada",

    F.from_unixtime(
        F.unix_timestamp(
            F.col("_fecha_base_timestamp")
        )
        +
        F.col("_segundo_inicio").cast("long")
    ).cast("timestamp")
)


# ============================================================
# 19. INICIO Y FIN DE LLAMADA
# ============================================================

df_split_1 = df_split_1.withColumn(
    "Inicio",
    F.col("Fecha_Hora_Llamada")
)

df_split_1 = df_split_1.withColumn(
    "Fin",

    F.from_unixtime(
        F.unix_timestamp(
            F.col("Inicio")
        )
        +
        F.col("segundos").cast("long")
    ).cast("timestamp")
)


# ============================================================
# 20. CREAR HORA Y TRAMA
# ============================================================

df_split_1 = df_split_1.withColumn(
    "hora",
    F.date_format(
        F.col("Inicio"),
        "HH:mm:ss"
    )
)

df_split_1 = df_split_1.withColumn(
    "Trama_Hora",
    F.hour(
        F.col("Inicio")
    )
)


# ============================================================
# 21. COLUMNAS DE ESTADO
# ============================================================

df_split_1 = df_split_1.withColumn(
    "Estados",
    F.lit("").cast("string")
)

df_split_1 = df_split_1.withColumn(
    "Sub_estado",
    F.lit("").cast("string")
)

df_split_1 = df_split_1.withColumn(
    "Codigo_Paleta",
    F.col("Codigo_Paleta").cast("string")
)


# ============================================================
# 22. LIMPIAR COLUMNAS AUXILIARES
# ============================================================

columnas_auxiliares = [
    "_prob_vdad_base",
    "_score_vdad",
    "_prob_vdad_final",
    "_rnd_vdad",
    "_duracion",
    "_id_temporal",
    "_ultimo_inicio",
    "_orden",
    "_gap",
    "_grupo_bot",
    "_duracion_grupo",
    "_gap_grupo",
    "_segundo_inicio",
    "_segundo_fin",
    "_fecha_base_timestamp"
]

columnas_a_eliminar = [
    columna
    for columna in columnas_auxiliares
    if columna in df_split_1.columns
]

df_split_1 = df_split_1.drop(
    *columnas_a_eliminar
)


# ============================================================
# 23. OPCIONAL: DEJAR UNA SOLA LLAMADA POR DNI
# ============================================================
# Descomenta únicamente si necesitas un registro por cliente.

# df_split_1 = df_split_1.dropDuplicates(["Dni"])


# ============================================================
# 24. ORDENAR RESULTADO
# ============================================================

df_split_1 = df_split_1.orderBy(
    F.col("Fecha_Hora_Llamada").asc(),
    F.col("DNI_Ejecutivo").asc()
)



In [41]:

# # ============================================================
# # 25. MOSTRAR RESULTADO
# # ============================================================

# df_split_1.select(
#     "Dni",
#     "DNI_Ejecutivo",
#     "Ejecutivo",
#     "Fecha_Hora_Llamada",
#     "segundos",
#     "Fecha_Llamada",
#     F.col('cl_telf1').alias("PHONE_NUMBER"),
#     "Codigo_Paleta",
#     "Inicio",
#     "Fin",
#     "Numero_Campana",
#     "Trama_Hora",
#     "list_name",
#     "list_description",
#     "Dentro_Rango_Horario",
#     "Estados",
#     "Sub_estado"
# ).show(
#     5,
#     truncate=False
# )

In [55]:

df_split_1 = df_split_1.withColumn(
    "Trama_Hora",
    F.hour(F.col("Inicio"))
)

In [14]:
df_split_1=df_split_1.dropDuplicates(['Dni'])

In [15]:
df_split_1 = df_split_1.withColumn(
    "Codigo_Paleta",
    F.col("Codigo_Paleta").cast("string")
)

In [58]:
segundos_random = (
    F.floor(F.rand() * (91 - 60 + 1)) + 60
)

In [ ]:
df_split_1=df_split_1.withColumn('DNI_Ejecutivo',F.lit('49587612')) 
df_split_1=df_split_1.withColumn('Ejecutivo',F.lit('Bot Bco Alfin')) 
df_split_1=df_split_1.withColumn('Codigo_Paleta',when(
        (F.col("correo_formulario")==1)|
        (F.col("correo_formulario")=='1')
    ,'zzz2').otherwise(F.col('Codigo_Paleta'))) 

In [60]:
df_split_1 = df_split_1.withColumn(
    "Fin",
    F.when(
        (F.col("correo_formulario")==1)|
        (F.col("correo_formulario")=='1')
        ,
        F.from_unixtime(
            F.unix_timestamp("Inicio") + segundos_random
        ).cast("timestamp")
    ).otherwise(F.col("Fin"))
)

In [61]:
df_split_1=df_split_1.withColumn('segundos',
    when(
        (F.col("correo_formulario")==1)|
        (F.col("correo_formulario")=='1')
        ,F.unix_timestamp(F.col("Fin")) - F.unix_timestamp(F.col("Inicio")))
    .otherwise(F.col('segundos'))
) 


In [74]:
df_split_1.select('Dni', 'DNI_Ejecutivo', 'Ejecutivo', 'Fecha_Hora_Llamada', 'segundos', 'Fecha_Llamada', 'Trama_Hora', F.col('celular').alias('PHONE_NUMBER'), 'Codigo_Paleta', 'Inicio', 'Fin','Numero_Campana','Trama_Hora','list_name','list_description','Nombre_Campana').show(3)

+--------+-------------+-------------+-------------------+--------+-------------+----------+------------+-------------+-------------------+-------------------+--------------+----------+-----------+----------------+--------------+
|     Dni|DNI_Ejecutivo|    Ejecutivo| Fecha_Hora_Llamada|segundos|Fecha_Llamada|Trama_Hora|PHONE_NUMBER|Codigo_Paleta|             Inicio|                Fin|Numero_Campana|Trama_Hora|  list_name|list_description|Nombre_Campana|
+--------+-------------+-------------+-------------------+--------+-------------+----------+------------+-------------+-------------------+-------------------+--------------+----------+-----------+----------------+--------------+
|00000158|     49587612|Bot Bco Alfin|2026-08-01 13:32:10|       0|   2026-08-01|        13|   938368849|        PDROP|2026-08-01 13:32:10|2026-08-01 13:32:10|           401|        13|provicional|     provicional|     BOT_ALFIN|
|00001285|     49587612|Bot Bco Alfin|2026-08-01 09:42:31|      44|   2026-08-01

In [17]:
df_split_1=df_split_1.withColumn('Sub_estado',F.lit(''))
df_split_1=df_split_1.withColumn('Estados',F.lit(''))

In [46]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

w = Window.partitionBy('PROPENSION_IC','FRESCURA','segmento',"USER_V3").orderBy(F.rand())

df_partes_alfin = df_split_1.withColumn("grupo_split", F.ntile(3).over(w))

df_parte_1 = df_partes_alfin.filter(F.col("grupo_split") == 1).drop("grupo_split")
df_parte_2 = df_partes_alfin.filter(F.col("grupo_split") == 2).drop("grupo_split")
df_parte_3 = df_partes_alfin.filter(F.col("grupo_split") == 3).drop("grupo_split")
# df_parte_4 = df_partes_alfin.filter(F.col("grupo_split") == 4).drop("grupo_split")
# df_parte_5 = df_partes_alfin.filter(F.col("grupo_split") == 5).drop("grupo_split")
# df_parte_6 = df_partes_alfin.filter(F.col("grupo_split") == 6).drop("grupo_split")
# df_parte_7 = df_partes_alfin.filter(F.col("grupo_split") == 7).drop("grupo_split")

print(
    df_parte_1.count(),
    df_parte_2.count(),
    df_parte_3.count()
    # df_parte_4.count()
    # df_parte_5.count(),
    # df_parte_6.count(),
    # df_parte_7.count()
)

3715 3692 3667


In [47]:
df_list_filtrada = df_split_1.toPandas()
ruta_archivo = os.path.join(ruta_csv, 'alfin_RE.xlsx')
df_list_filtrada.to_excel(ruta_archivo, index=False)

In [18]:
df_split_1=df_split_1.select('Dni', 'DNI_Ejecutivo', 'Ejecutivo', 'Fecha_Hora_Llamada', 'segundos', 'Fecha_Llamada', F.col('celular').alias('PHONE_NUMBER'), 'Codigo_Paleta', 'Inicio', 'Fin','Numero_Campana','Trama_Hora','list_name','list_description','Estados','Sub_estado','Nombre_Campana')

In [49]:
df_split_1.show(3)

+--------+-------------+-------------+-------------------+--------+-------------+------------+-------------+-------------------+-------------------+--------------+----------+-----------+----------------+-------+----------+--------------+
|     Dni|DNI_Ejecutivo|    Ejecutivo| Fecha_Hora_Llamada|segundos|Fecha_Llamada|PHONE_NUMBER|Codigo_Paleta|             Inicio|                Fin|Numero_Campana|Trama_Hora|  list_name|list_description|Estados|Sub_estado|Nombre_Campana|
+--------+-------------+-------------+-------------------+--------+-------------+------------+-------------+-------------------+-------------------+--------------+----------+-----------+----------------+-------+----------+--------------+
|00085951|     49587612|Bot Bco Alfin|2026-08-08 13:48:28|       0|   2026-08-08|   961968635|           NA|2026-08-08 13:48:28|2026-08-08 13:48:28|           401|        13|provicional|     provicional|       |          |     BOT_ALFIN|
|00096179|     49587612|Bot Bco Alfin|2026-08-08

In [66]:
df_split_1.count()

10101

In [ ]:
# print([row['Fecha_Hora_Llamada' ] for row in df_split.select('Fecha_Hora_Llamada').distinct().collect()])

[datetime.datetime(2026, 7, 23, 5, 18, 55), datetime.datetime(2026, 7, 23, 6, 55, 51), datetime.datetime(2026, 7, 23, 8, 3, 57), datetime.datetime(2026, 7, 23, 7, 55, 8), datetime.datetime(2026, 7, 23, 6, 44, 53), datetime.datetime(2026, 7, 23, 4, 14, 29), datetime.datetime(2026, 7, 23, 8, 22, 14), datetime.datetime(2026, 7, 23, 4, 38, 34), datetime.datetime(2026, 7, 23, 5, 57, 7), datetime.datetime(2026, 7, 23, 7, 39, 56), datetime.datetime(2026, 7, 23, 4, 5, 59), datetime.datetime(2026, 7, 23, 7, 33, 25), datetime.datetime(2026, 7, 23, 4, 43, 57), datetime.datetime(2026, 7, 23, 5, 38, 29), datetime.datetime(2026, 7, 23, 7, 47, 55), datetime.datetime(2026, 7, 23, 5, 2, 53), datetime.datetime(2026, 7, 23, 8, 28, 43), datetime.datetime(2026, 7, 23, 8, 29, 50), datetime.datetime(2026, 7, 23, 4, 32, 17), datetime.datetime(2026, 7, 23, 5, 17, 18), datetime.datetime(2026, 7, 23, 6, 16, 7), datetime.datetime(2026, 7, 23, 5, 26, 19), datetime.datetime(2026, 7, 23, 7, 7, 16), datetime.datetime

In [20]:
df_split_1=df_split_1.dropDuplicates(['Dni'])
df_split_1.count()

15000

In [21]:
append_table_SQL(spark,df_split_1,f'tmp_llamadas_mes',server_zeus,user_zeus,pwd_zeus,'SAMANTHA')


In [65]:
df_split_1.show(3)

+--------+-------------+-------------+-------------------+--------+-------------+------------+-------------+-------------------+-------------------+--------------+----------+-----------+----------------+-------+----------+--------------+
|     Dni|DNI_Ejecutivo|    Ejecutivo| Fecha_Hora_Llamada|segundos|Fecha_Llamada|PHONE_NUMBER|Codigo_Paleta|             Inicio|                Fin|Numero_Campana|Trama_Hora|  list_name|list_description|Estados|Sub_estado|Nombre_Campana|
+--------+-------------+-------------+-------------------+--------+-------------+------------+-------------+-------------------+-------------------+--------------+----------+-----------+----------------+-------+----------+--------------+
|00000158|     49587612|Bot Bco Alfin|2026-08-14 12:37:04|       0|   2026-08-14|   938368849|           AB|2026-08-14 12:37:04|2026-08-14 12:37:04|           401|        12|provicional|     provicional|       |          |     BOT_ALFIN|
|00002331|     49587612|Bot Bco Alfin|2026-08-14

In [210]:
print([row['estado' ] for row in df_split.select('estado').distinct().collect()])


['ENVIADO   ', None]


In [38]:
df_necesito.show(2)
# df_necesito=df_necesito.withColumn('Numero_Campana',F.lit('401'))
# df_necesito=df_necesito.withColumn('list_description',F.lit('provicional'))
# df_necesito=df_necesito.withColumn('list_name',F.lit('provicional'))
# df_necesito=df_necesito.withColumn('Nombre_Campana',F.lit('BOT_ALFIN'))
# df_necesito=df_necesito.withColumn('Nombre_Campana',F.lit('2026-07 BCO ALFIN'))


+--------+-------------+------------------+-------------------+--------+-------------+----------+------------+-------------+-------------------+-------------------+
|     Dni|DNI_Ejecutivo|         Ejecutivo| Fecha_Hora_Llamada|segundos|Fecha_Llamada|Trama_Hora|PHONE_NUMBER|Codigo_Paleta|             Inicio|                Fin|
+--------+-------------+------------------+-------------------+--------+-------------+----------+------------+-------------+-------------------+-------------------+
|72048296|         VDAD|Outbound Auto Dial|2023-11-24 17:34:47|       0|   2023-11-24|        17|   922488467|           AB|2023-11-24 17:34:47|2023-11-24 17:34:47|
|29690194|         VDAD|Outbound Auto Dial|2023-11-24 17:34:47|       0|   2023-11-24|        17|   983941534|           AB|2023-11-24 17:34:47|2023-11-24 17:34:47|
+--------+-------------+------------------+-------------------+--------+-------------+----------+------------+-------------+-------------------+-------------------+
only showi

In [205]:
df_split.show(5)

+--------+-------------+--------+--------------------+---------+----+------------+------------+-----------+-----------+-------------------+--------------+------+-------+--------------+----------------+-----------+-----------------+-------------+-------------+------------------+-------------+
|     Dni|PROPENSION_IC|FRESCURA|             USER_V3|TIPO_LOTE|lote|PHONE_NUMBER|fecha_visita|hora_visita|fecha_envio|intentos_realizados|fecha_registro|estado|por_dia|Numero_Campana|list_description|  list_name|   Nombre_Campana|Fecha_Llamada|DNI_Ejecutivo|         Ejecutivo|Codigo_Paleta|
+--------+-------------+--------+--------------------+---------+----+------------+------------+-----------+-----------+-------------------+--------------+------+-------+--------------+----------------+-----------+-----------------+-------------+-------------+------------------+-------------+
|47319495|            1|       0|2. sunedu & sunarp B| BASE BOT| BOT|   954821826|        NULL|       NULL|       NULL|  

In [ ]:


Bot Bco Alfin



In [186]:

query = """
select top(10)* From [THOTH].dbo.Tmp_LLamadas_Alfin_bot
    """
df_llamadas_bot=obtener_tabla_sql(spark,query,server_zeus,user_zeus,pwd_zeus,db_zeus)
df_llamadas_bot.show()

+--------+--------------+--------------+-------------+------------------+-------------------+--------+-------------+----------+-------+----------+--------------------+----------------+----------------+------------+------------+-----------+-------------+-------------------+-------------------+--------+--------+--------------------+--------------------+-----+------------------+----------+------------+---+-------+
|     Dni|Numero_Campana|Nombre_Campana|DNI_Ejecutivo|         Ejecutivo| Fecha_Hora_Llamada|segundos|Fecha_Llamada|Trama_Hora|Estados|Sub_estado|         Descripcion|list_description|       list_name|PHONE_NUMBER|Fecha_Agenda|Comentarios|Codigo_Paleta|             Inicio|                Fin| lead_id| Estado_|         Sub_Estado_|        Descripcion_|Pesos|            Enlace|Fecha_Llam|Hora_Llamada| RH|COD_BCO|
+--------+--------------+--------------+-------------+------------------+-------------------+--------+-------------+----------+-------+----------+--------------------+---

In [187]:
print([row['DNI_Ejecutivo' ] for row in df_llamadas_bot.select('DNI_Ejecutivo').distinct().collect()])


['49587612', 'VDAD']


In [101]:
print(41539-41668)

-129


In [ ]:
nocet 41668
1760 odo
bot 43428

In [58]:
49587612

['ACTIVO', 'RETIVO_CORREO']


In [2]:
print(86857*.50,'bot')
print((102599+2779)*.3,'hum')
print(192235)

43428.5 bot
31613.399999999998 hum
192235


In [23]:
print([row['estado' ] for row in df_crm.select('estado').distinct().collect()])
# print([row['codigo_ejecutivo_id' ] for row in df_crm.select('codigo_ejecutivo_id').distinct().collect()])


['PENDIENTE ', 'ERROR     ', 'ENVIADO   ']


In [ ]:
['NUMERO_DOCUMENTO', 'NRG', 'Dni', 'TIPO_GESTION', 'GESTION', 'SUBGESTION', 'TIPO_GESTION_hum', 'GESTION_hum', 'SUBGESTION_hum', 'TIPO_GESTION_bot', 'GESTION_bot', 'SUBGESTION_bot', 'TIPO', 'RECORRIDO', 'RECORRIDO_hum', 'RECORRIDO_bot', 'CET', 'CET_hum', 'CET_bot', 'CONT_GEN', 'CONT_GEN_hum', 'CONT_GEN_bot', 'Hora_Llamada', 'Mejor_Telefono', 'TELEFONO', 'FECHA_LLAMADA', 'DIA', 'Ejecutivo', 'RHFC', 'SUPERVISOR', 'segundos', 'AGENDADOS', 'AGENDADOS_hum', 'AGENDADOS_bot', 'SOLO FH1', 'SOLO FH2', 'SOLO FH3', 'DOBLE FH', 'TRIPLE FH', 'NUMXDNI', 'Estado', 'CntEstado', 'MntOferta', 'CNTVTAS', 'NUM_DIA_HABIL', 'Semana_Mes', 'tMontoDesem', 'CNT_LLAMADAS', 'CNT_LLAMADAS_hum', 'CNT_LLAMADAS_bot', 'AÑO_DURACION_BASE', 'MES_DURACION_BASE', 'FECHA_ENVIO', 'SERVICIO', 'RETIRO', 'REP1', 'REP2', 'FLG_SIN_ENR', 'tMesGestion', 'DESCRIPCION', 'DESCRIPCION2', 'llave', 'DEPARTAMENTO', 'DISTRITO', 'PROPENSION_IC', 'CUOTA', 'Agencia_comercial', 'Region_comercial', 'color_final', 'GRUPO_TASA', 'GRUPO_MONTO', 'tipo_cliente_riegos', 'USER_V3', 'TIPO_CLIENTE', 'FRESCURA', 'TIPO_BASE', 'campania', 'FLG_AAHH', 'INTENSIDAD_MAX', 'REGION', 'RANGO_EDAD', 'RANGO_OFERTA', 'RANGO_TASA', 'TIPO_LOTE', 'TIPO_TELF', 'lote', 'tMontoDesemFugas']prop

['NUMERO_DOCUMENTO', 'NRG', 'Dni', 'TIPO_GESTION', 'GESTION', 'SUBGESTION', 'TIPO_GESTION_hum', 'GESTION_hum', 'SUBGESTION_hum', 'TIPO_GESTION_bot', 'GESTION_bot', 'SUBGESTION_bot', 'TIPO', 'RECORRIDO', 'RECORRIDO_hum', 'RECORRIDO_bot', 'CET', 'CET_hum', 'CET_bot', 'CONT_GEN', 'CONT_GEN_hum', 'CONT_GEN_bot', 'Hora_Llamada', 'Mejor_Telefono', 'TELEFONO', 'FECHA_LLAMADA', 'DIA', 'Ejecutivo', 'RHFC', 'SUPERVISOR', 'segundos', 'AGENDADOS', 'AGENDADOS_hum', 'AGENDADOS_bot', 'SOLO FH1', 'SOLO FH2', 'SOLO FH3', 'DOBLE FH', 'TRIPLE FH', 'NUMXDNI', 'Estado', 'CntEstado', 'MntOferta', 'CNTVTAS', 'NUM_DIA_HABIL', 'Semana_Mes', 'tMontoDesem', 'CNT_LLAMADAS', 'CNT_LLAMADAS_hum', 'CNT_LLAMADAS_bot', 'AÑO_DURACION_BASE', 'MES_DURACION_BASE', 'FECHA_ENVIO', 'SERVICIO', 'RETIRO', 'REP1', 'REP2', 'FLG_SIN_ENR', 'tMesGestion', 'DESCRIPCION', 'DESCRIPCION2', 'llave', 'DEPARTAMENTO', 'DISTRITO', 'PROPENSION_IC', 'CUOTA', 'Agencia_comercial', 'Region_comercial', 'color_final', 'GRUPO_TASA', 'GRUPO_MONTO', '

In [15]:
query = """
    SELECT * FROM maeba.[ADM_OBJ_TG].[tGestionMesAlfin]
    where RECORRIDO = 0
    """
df_recorrido=obtener_tabla_sql(spark,query,server_zeus,user_zeus,pwd_zeus,db_zeus)
df_recorrido.count()

156930

In [6]:
df_formato.join(df_recorrido,['NUMERO_DOCUMENTO'],'inner').count()

0

In [29]:
df_recorrido.show()

+----------------+---+----+------------+----------+----------+----------------+-----------+--------------+----------------+-----------+--------------+----------+---------+-------------+-------------+---+-------+-------+--------+------------+------------+------------+--------------+--------+-------------+----+---------+----+-----------+--------+---------+-------------+-------------+--------+--------+--------+--------+---------+-------+------+---------+---------+-------+-------------+----------+-----------+------------+----------------+----------------+-----------------+-----------------+-----------+--------+------+-----+-----+-----------+-----------+--------------------+------------------+-------------------+------------+--------------------+-------------+-------+--------------------+--------------------+---------------+-------------+---------------+-------------------+--------------------+-------------+--------+----------------+--------+------------+--------------+---------------+-----

In [31]:
df_recorrido.count()

1999

In [33]:
# print([row['FEC_NAC' ] for row in df_list.select('FEC_NAC').distinct().collect()])
df_recorrido.groupBy('PROPENSION_IC') \
    .count() \
    .orderBy('PROPENSION_IC') \
    .show(30)


+-------------+-----+
|PROPENSION_IC|count|
+-------------+-----+
|            1|  572|
|            2|  980|
|            3|  132|
|            4|  122|
|            5|  116|
|            6|   77|
+-------------+-----+



In [ ]:
df_recorrido

In [25]:
query = """
    select top(10)* From SAMANTHA..tmp_llamadas_mes
    """
df_recorrido=obtener_tabla_sql(spark,query,server_zeus,user_zeus,pwd_zeus,db_zeus)
df_recorrido.show(2)

+--------+--------------+--------------------+-------------+------------------+-------------------+--------+-------------+----------+-------+----------+-----------+--------------------+--------------------+------------+------------+-----------+-------------+-------------------+-------------------+-------+
|     Dni|Numero_Campana|      Nombre_Campana|DNI_Ejecutivo|         Ejecutivo| Fecha_Hora_Llamada|segundos|Fecha_Llamada|Trama_Hora|Estados|Sub_estado|Descripcion|    list_description|           list_name|PHONE_NUMBER|Fecha_Agenda|Comentarios|Codigo_Paleta|             Inicio|                Fin|lead_id|
+--------+--------------+--------------------+-------------+------------------+-------------------+--------+-------------+----------+-------+----------+-----------+--------------------+--------------------+------------+------------+-----------+-------------+-------------------+-------------------+-------+
|72048296|            80|2023-11 CENCOSUD ...|         VDAD|Outbound Auto Dial|

In [ ]:
RetiroDefinitivo_BlackList